In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ishanshrivastava28/tata-online-retail-dataset")
print("Path to dataset files:", path)

file_path = path + '/Online Retail Data Set.csv'

import pandas as pd
import numpy as np

data = pd.read_csv(file_path, encoding='latin1')
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], dayfirst=True)

# -----------------------------
# Minimal cleaning (portfolio-safe)
# -----------------------------
data = data.dropna(subset=['CustomerID'])
data = data[(data['Quantity'] > 0) & (data['UnitPrice'] > 0)]
data['CustomerID'] = data['CustomerID'].astype(int)

data['amount'] = data['UnitPrice'] * data['Quantity']
data = data[['CustomerID', 'InvoiceDate', 'amount']]

data['event_month'] = data['InvoiceDate'].dt.to_period('M')

data['cohort_month'] = (
    data.groupby('CustomerID')['event_month']
    .transform('min')
)

data['cohort_index'] = (
    (data['event_month'].dt.year - data['cohort_month'].dt.year) * 12
    + (data['event_month'].dt.month - data['cohort_month'].dt.month)
)

cohort_counts = (
    data.groupby(['cohort_month', 'cohort_index'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'active_users'})
)

cohort_revenue = (
    data.groupby(['cohort_month', 'cohort_index'])['amount']
    .sum()
    .reset_index()
)

cohort_table = cohort_counts.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='active_users'
)

cohort_revenue_table = cohort_revenue.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='amount'
)

base = cohort_table.iloc[:, 0]
cohort_table = cohort_table.divide(base, axis=0)

retention = cohort_table.round(3) * 100
retention.index = retention.index.astype(str)

retention = retention.fillna(0.0)
cohort_revenue_table = cohort_revenue_table.fillna(0.0)

# Optional: keep only first 12 months for readability
# retention = retention.iloc[:, :13]
# cohort_revenue_table = cohort_revenue_table.iloc[:, :13]

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

data_for_heatmap = retention.values
im = ax.imshow(data_for_heatmap, aspect='auto', cmap='YlGnBu')

ax.set_xlabel('Cohort Index (Months)')
ax.set_ylabel('Cohort Month')

ax.set_xticks(np.arange(len(retention.columns)))
ax.set_yticks(np.arange(len(retention.index)))

ax.set_xticklabels(retention.columns)
ax.set_yticklabels(retention.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for i in range(len(retention.index)):
    for j in range(len(retention.columns)):
        ax.text(j, i, f'{data_for_heatmap[i, j]:.1f}',
                ha='center', va='center', color='black', fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Retention (%)')

plt.title('Customer Retention Cohort Heatmap')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))

data_for_heatmap = cohort_revenue_table.values
im = ax.imshow(data_for_heatmap, aspect='auto', cmap='Oranges')

ax.set_xlabel('Cohort Index (Months)')
ax.set_ylabel('Cohort Month')

ax.set_xticks(np.arange(len(cohort_revenue_table.columns)))
ax.set_yticks(np.arange(len(cohort_revenue_table.index)))

ax.set_xticklabels(cohort_revenue_table.columns)
ax.set_yticklabels(cohort_revenue_table.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for i in range(len(cohort_revenue_table.index)):
    for j in range(len(cohort_revenue_table.columns)):
        ax.text(j, i, f'{data_for_heatmap[i, j]:.1f}',
                ha='center', va='center', color='black', fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Revenue Retention ($)')

plt.title('Revenue Retention Cohort Heatmap')
plt.tight_layout()
plt.show()
